# Generate Synthetic Dataset using LLM and ACL Anthology Corpus to finetune embedding model
In this notebook, a synthetic dataset of (query, relevant documents) pairs are generated from a corpus of documents without labelers by leveraging LLM.
## Setting up Environment

In [29]:
!git clone https://github.com/run-llama/finetune-embedding.git

fatal: destination path 'finetune-embedding' already exists and is not an empty directory.


In [30]:
!pip install -r /kaggle/working/finetune-embedding/requirements.txt
!pip install groq
!pip install pipreqs

In [31]:
%%writefile .env
GROQ_API_KEY='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'

Overwriting .env


In [41]:
%%writefile .gitignore
.env

Overwriting .gitignore


In [42]:
!mkdir data

In [33]:
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from dotenv import load_dotenv, find_dotenv
from typing import Literal 
from groq import Groq
import re
import pandas as pd
import json
import uuid
import os

## Load and Split Corpus
The cleaned and chunked corpus is loaded, split into training and validation sets, and saved as JSON files

In [43]:
CORPUS_PATH = '/kaggle/input/sub-chunk-kb-acl/sub_chunk_kb_acl-100k.csv'
TRAIN_CORPUS_PATH = "data/train_corpus.json"
VAL_CORPUS_PATH = "data/val_corpus.json"

In [44]:
def split_and_save_dataset(input_file,  train_output_file, val_output_file, test_size=0.2, random_state=42, head_value=None):
    """
    Splits the dataset into training and validation sets and saves them as JSON files.

    Args:
        input_file (str): Path to the input CSV file containing the dataset.
        train_output_file (str): Path to save the training set as a JSON file.
        val_output_file (str): Path to save the validation set as a JSON file.
        test_size (float, optional): The proportion of the dataset to include in the validation split.
                                      Defaults to 0.2 (20%).
        random_state (int, optional): Controls the shuffling applied to the data before splitting.
                                      Defaults to 42.
        head_value (int, optional): ###

    Returns:
        None
    """
    # Read CSV file into pandas DataFrame
    df = pd.read_csv(input_file)
    
    # Testing with only first 30 rows
    
    if head_value:
        df = df.head(head_value)

    # Drop the title column if it exists
    if "title" in df.columns:
        df.drop(columns=["title"], inplace=True)

    # Split the dataset into training and validation sets
    train_df, val_df = train_test_split(df, test_size=test_size, random_state=random_state)

    # Convert DataFrame columns to lists
    train_data = {str(uuid.uuid4()): text for text in train_df["text"].tolist()}
    val_data = {str(uuid.uuid4()): text for text in val_df["text"].tolist()}

    # Save the training and validation sets as JSON files
    with open(train_output_file, "w") as train_json_file:
        json.dump(train_data, train_json_file)

    with open(val_output_file, "w") as val_json_file:
        json.dump(val_data, val_json_file)

    print("Training and validation sets saved successfully.")


In [45]:
split_and_save_dataset(input_file=CORPUS_PATH , train_output_file=TRAIN_CORPUS_PATH, val_output_file=VAL_CORPUS_PATH, head_value=30)

Training and validation sets saved successfully.


In [46]:
with open(TRAIN_CORPUS_PATH, 'r+') as f:
    train_corpus = json.load(f)

with open(VAL_CORPUS_PATH, 'r+') as f:
    val_corpus = json.load(f)
    

## Generate Synthetic Queries


In [47]:
TRAIN_QUERIES_PATH = 'data/train_queries.json'
TRAIN_RELEVANT_DOCS_PATH = 'data/train_relevant_docs.json'

VAL_QUERIES_PATH = 'data/val_queries.json'
VAL_RELEVANT_DOCS_PATH = 'data/val_relevant_docs.json'

### Prepare LLM. Options: Llama, Gemma, Mixtral, accessed via groq

#### llm() function is created to generate questions/queries using each text chunk in the corpus as context.

Each pair of (generated question, text chunk used as context) becomes a datapoint in the finetuning dataset (either for training or evaluation).

In [48]:
load_dotenv(find_dotenv())

CHAT_MODEL = Literal["llama3-8b-8192", "llama3-70b-8192", "mixtral-8x7b-32768", "gemma-7b-it"]
QUERY_GEN_TEMPLATE = """\
    Generate one question based on the following context: \
    {context}. Return only the question string without any prefix \
    The question length should not exceed 30 words.
    """


groq_api_key = os.environ["GROQ_API_KEY"]
client = Groq(
    api_key=groq_api_key,
)

def llm(
    context: str,
    preamble: str,
    model: str = "mixtral-8x7b-32768",
    temperature: float = 0.5,
    max_tokens: int = 1024,
    top_p: float = 1.0,
    stop: list[str] | None = None,
    stream: bool = False,
) -> str:
   
    chat_completion = client.chat.completions.create(
        messages=[
            {
                "role": "system",
                "content": preamble
            },
            {
                "role": "user",
                "content": context,
            }
        ],
        model=model,
        temperature=temperature,
        max_tokens=max_tokens,
        top_p=top_p,
        stop=stop,
        stream=stream,
    )

    return chat_completion.choices[0].message.content

In [49]:
def generate_queries(
    corpus,
    prompt_template=None,
):
    """
    Automatically generate hypothetical questions that could be answered with
    doc in the corpus using the llm function.
    """
    queries = {}
    relevant_docs = {}
    for chunk_id, text in tqdm(corpus.items()):
        preamble = prompt_template.format(context=text)
        response = llm(context=text, preamble=preamble)
 
        result = str(response).strip().split("\n")
        questions = [
            re.sub(r"^\d+[\).\s]", "", question).strip() for question in result
        ]
        questions = [question for question in questions if len(question) > 0]
        
        for question in questions:
            question_id = str(uuid.uuid4())
            queries[question_id] = question
            relevant_docs[question_id] = [chunk_id]
    return queries, relevant_docs

In [50]:
train_queries, train_relevant_docs = generate_queries(train_corpus, prompt_template=QUERY_GEN_TEMPLATE)

100%|██████████| 24/24 [00:36<00:00,  1.51s/it]


In [51]:
val_queries, val_relevant_docs = generate_queries(val_corpus, prompt_template=QUERY_GEN_TEMPLATE)

100%|██████████| 6/6 [00:21<00:00,  3.54s/it]


In [52]:
with open(TRAIN_QUERIES_PATH, 'w+') as f:
    json.dump(train_queries, f)

with open(TRAIN_RELEVANT_DOCS_PATH, 'w+') as f:
    json.dump(train_relevant_docs, f)

with open(VAL_QUERIES_PATH, 'w+') as f:
    json.dump(val_queries, f)

with open(VAL_RELEVANT_DOCS_PATH, 'w+') as f:
    json.dump(val_relevant_docs, f)

### Merge data
Finally, some minor re-organization is done to make it easier to access the dataset for training and evaluation

In [53]:
TRAIN_DATASET_PATH = 'data/train_dataset.json'
VAL_DATASET_PATH = 'data/val_dataset.json'

In [54]:
train_dataset = {
    'queries': train_queries,
    'corpus': train_corpus,
    'relevant_docs': train_relevant_docs,
}

val_dataset = {
    'queries': val_queries,
    'corpus': val_corpus,
    'relevant_docs': val_relevant_docs,
}

In [55]:
with open(TRAIN_DATASET_PATH, 'w+') as f:
    json.dump(train_dataset, f)

with open(VAL_DATASET_PATH, 'w+') as f:
    json.dump(val_dataset, f)

In [56]:
!pipreqs